# 0067: the view reader's read, on a Colab card

The answers only: each instruction read whole by the adapter, its answer held token by token to the grammar (rules
1-9, `--grammar typed`), the same rows as the read at home (each split's seeded sample). The VM scores the answers at
home. A batch of 32 instructions decodes in lockstep; about ten minutes for 300 rows on an A100 (the local card: two
hours).

**Set `ADAPTER` below** (`cubby450m_view_reader_v1` or `_v2`). It reads, from Drive, the base, the tokenizer, the
adapter and `rules/long_rows_0067/rows.jsonl`; it writes `rules/reads/<adapter>_<grammar>_answers.jsonl` (resumable:
a rerun keeps the rows already answered). **Disconnect when it prints `read:`.**

In [ ]:
import os, subprocess, sys
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!pip -q install tokenizers safetensors numpy regex
from google.colab import drive; drive.mount('/content/drive')
REPO, DRIVE = '/content/CubbyLLM', '/content/drive/MyDrive/cubbyllm'
ADAPTER   = 'cubby450m_view_reader_v2'          # or 'cubby450m_view_reader_v1'
GRAMMAR, SPLITS, N, BATCH = 'typed', 'dev,test,test_long', 100, 32
CKPT      = f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt'
TOKENIZER = f'{DRIVE}/token_cache_cont/tokenizer/grillcheese_bbpe128k.json'
ROWS      = f'{DRIVE}/rules/long_rows_0067/rows.jsonl'
ADIR      = f'{DRIVE}/rules/{ADAPTER}'
OUT       = f'{DRIVE}/rules/reads/{ADAPTER}_{GRAMMAR}_answers.jsonl'
for p in (CKPT, TOKENIZER, ROWS, f'{ADIR}/talk_lora.safetensors', f'{ADIR}/talk_lora.json'):
    assert os.path.exists(p), p
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
os.environ['CB_NO_FLEX'] = '1'
cmd = [sys.executable, '-u', 'validation/exp_view_reader_torch.py', '--ckpt', CKPT, '--tokenizer', TOKENIZER,
       '--adapter', ADIR, '--rows', ROWS, '--splits', SPLITS, '--n', str(N), '--batch', str(BATCH),
       '--grammar', GRAMMAR, '--out', OUT]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
assert p.wait() == 0, f'the read failed ({p.returncode})'
!wc -l "{OUT}"

## Done

**Disconnect the runtime.** At home, Claude Code scores the answers in the VM:
`exp_view_reader.py --score <answers> --splits dev` and `--splits test,test_long` (lines 1-4), then the second
chance on the dev answers.